In [47]:
import pandas as pd
import numpy as np

# 1. Opis skupa podataka
**Naziv skupa podataka:** LinkedIn Job Postings

**Link:** https://www.kaggle.com/datasets/arshkon/linkedin-job-postings

**Tip zadatka:** Klasterovanje

**Format podataka:** CSV

**Broj tabela:** 11

**Glavna tabela:** `postings.csv`

**Broj oglasa za posao:** 123849

**Period prikupljanja podataka:** 2023–2024

## Kratak opis

LinkedIn Job Postings predstavlja skup podataka koji sadrzi vise od 123 hiljade oglasa za posao objavljenih na LinkedIn platformi tokom 2023. i 2024. godine. Pored osnovnih informacija o poslovima, skup sadrzi dodatne tabele sa informacijama o kompanijama, platama, beneficijama, industrijama i vestinama.

Skup podataka je organizovan kao relacioni model sastavljen od 11 medjusobno povezanih tabela. Centralni entitet predstavlja oglas za posao, dok ostale tabele obogacuju podatke dodatnim informacijama o kompanijama i karakteristikama radnih mesta.

## Dimenzije tabela

| Tabela | Broj redova | Broj kolona |
|----------|----------:|----------:|
| postings | 123849 | 31 |
| companies | 24473 | 10 |
| benefits | 67943 | 3 |
| company_industries | 24375 | 2 |
| company_specialities | 169387 | 2 |
| employee_counts | 35787 | 4 |
| industries | 422 | 2 |
| job_industries | 164808 | 2 |
| job_skills | 213768 | 2 |
| salaries | 40785 | 8 |
| skills | 35 | 2 |


In [48]:
postings = pd.read_csv('../data/postings.csv')

## 2.1 Tabela postings

Tabela `postings` predstavlja centralnu tabelu skupa podataka. Svaki red odgovara jednom oglasu za posao objavljenom na LinkedIn platformi.

Tabela sadrzi 123849 oglasa i 31 atribut.

Najvaznije kolone mogu se podeliti u nekoliko logickih grupa:

### Identifikatori

- job_id
- company_id

Ove kolone sluze za povezivanje sa ostalim tabelama i nece biti koriscene kao ulazne karakteristike za klasterovanje.

### Informacije o poziciji

- title
- description
- skills_desc
- formatted_experience_level

Ove kolone opisuju samu poziciju i predstavljaju jedne od najvaznijih izvora informacija za analizu.

### Informacije o zaposlenju

- formatted_work_type
- application_type
- remote_allowed
- sponsored

Opisuje nacin rada i tip zaposlenja.

### Informacije o plati

- min_salary
- med_salary
- max_salary
- normalized_salary
- pay_period
- compensation_type
- currency

Ove kolone opisuju finansijske karakteristike oglasa.

### Informacije o popularnosti oglasa

- views
- applies

Predstavljaju broj pregleda i prijava za posao.

### Vremenske informacije

- listed_time
- original_listed_time
- expiry
- closed_time

Omogucavaju analizu zivotnog ciklusa oglasa.

In [49]:
postings["formatted_experience_level"].value_counts()

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

### Nivoi iskustva

Kolona `formatted_experience_level` sadrzi sest razlicitih kategorija nivoa iskustva:

| Nivo iskustva | Broj oglasa |
|----------|----------:|
| Mid-Senior level | 41489 |
| Entry level | 36708 |
| Associate | 9826 |
| Director | 3746 |
| Internship | 1449 |
| Executive | 1222 |

Najveci deo oglasa pripada kategorijama Mid-Senior i Entry level, što ukazuje da se najveci broj otvorenih pozicija odnosi na pocetne i srednje nivoe karijere.

In [50]:
companies = pd.read_csv('../data/companies.csv')

## 2.2 Tabela companies

Tabela `companies` sadrzi osnovne informacije o kompanijama koje objavljuju oglase za posao na LinkedIn platformi.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| company_id | Jedinstveni identifikator kompanije |
| name | Naziv kompanije |
| description | Opis kompanije |
| company_size | Velicina kompanije |
| state | Drzava/provincija |
| country | Drzava |
| city | Grad |
| zip_code | Postanski broj |
| address | Adresa kompanije |
| url | URL stranica kompanije |

### Podela atributa

#### Identifikatori

- company_id

Kolona sluzi za povezivanje sa ostalim tabelama i nema direktnu analiticku vrednost za klasterovanje.

#### Opisne karakteristike

- name
- description
- company_size

Ove kolone opisuju osnovne karakteristike kompanije i mogu biti korisne za dodatnu analizu ili prosirivanje glavne tabele.

#### Lokacione informacije

- country
- state
- city
- zip_code
- address

Kolone opisuju geografsku lokaciju kompanije.

#### Tehnicke informacije

- url

Predstavlja internet adresu kompanije i nema znacaj za proces klasterovanja.

In [51]:
benefits = pd.read_csv('../data/benefits.csv')

In [52]:
benefits["type"].unique()

array(['Medical insurance', 'Vision insurance', 'Dental insurance',
       '401(k)', 'Student loan assistance', 'Tuition assistance',
       'Disability insurance', 'Paid maternity leave',
       'Paid paternity leave', 'Child care support', 'Commuter benefits',
       'Pension plan'], dtype=object)

In [53]:
benefits["type"].value_counts()

type
401(k)                     24231
Medical insurance           9873
Vision insurance            9309
Disability insurance        7930
Dental insurance            6868
Tuition assistance          2614
Commuter benefits           2226
Paid maternity leave        1808
Paid paternity leave        1540
Pension plan                 906
Student loan assistance      365
Child care support           273
Name: count, dtype: int64

## 2.3 Tabela benefits

Tabela `benefits` sadrzi informacije o dodatnim pogodnostima (beneficijama) koje poslodavci nude uz odredjene pozicije.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| job_id | Identifikator oglasa za posao |
| inferred | Oznacava da li je beneficija eksplicitno navedena ili izvedena iz opisa |
| type | Tip beneficije |

### Podela atributa

#### Identifikatori

- job_id

Kolona sluzi za povezivanje sa tabelom `postings`.

#### Karakteristike beneficija

- inferred
- type

Ove kolone opisuju beneficije koje su povezane sa konkretnim oglasom za posao.

### Razlicite beneficije

U skupu podataka identifikovano je 12 razlicitih tipova beneficija:

| Beneficija | Broj pojavljivanja |
|----------|----------:|
| 401(k) | 24231 |
| Medical insurance | 9873 |
| Vision insurance | 9309 |
| Disability insurance | 7930 |
| Dental insurance | 6868 |
| Tuition assistance | 2614 |
| Commuter benefits | 2226 |
| Paid maternity leave | 1808 |
| Paid paternity leave | 1540 |
| Pension plan | 906 |
| Student loan assistance | 365 |
| Child care support | 273 |

 S obzirom da postoji samo 12 razlicitih tipova beneficija, moguce ih je jednostavno transformisati u binarne atribute (one-hot encoding) i koristiti kao ulazne karakteristike prilikom klasterovanja poslova.

In [54]:
company_industries = pd.read_csv('../data/company_industries.csv')

In [55]:
company_industries["industry"].nunique()

144

## 2.4 Tabela company_industries

Tabela `company_industries` povezuje kompanije sa industrijama u kojima posluju. Svaki red predstavlja pripadnost jedne kompanije odredjenoj industriji.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| company_id | Identifikator kompanije |
| industry | Naziv industrije |

### Podela atributa

#### Identifikatori

- company_id

Kolona sluzi za povezivanje sa tabelom `companies`.

#### Karakteristike kompanije

- industry

Predstavlja industriju u kojoj kompanija posluje.

### Analiza industrija

U skupu podataka identifikovane su:

- 24375 kompanija povezanih sa industrijama
- 144 razlicite industrije

In [56]:
company_specialities = pd.read_csv('../data/company_specialities.csv')

In [57]:
company_specialities["speciality"].nunique()

82960

## 2.5 Tabela company_specialities

Tabela `company_specialities` sadrzi informacije o oblastima specijalizacije kompanija. Jedna kompanija moze imati vise specijalnosti, zbog cega se ista kompanija moze pojaviti u vise redova.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| company_id | Identifikator kompanije |
| speciality | Oblast specijalizacije kompanije |

U tabeli je identifikovano 82960 razlicitih specijalnosti.

In [58]:
employee_counts = pd.read_csv('../data/employee_counts.csv')

## 2.6 Tabela employee_counts

Tabela `employee_counts` sadrzi informacije o velicini i popularnosti kompanija na LinkedIn platformi.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| company_id | Identifikator kompanije |
| employee_count | Broj zaposlenih u kompaniji |
| follower_count | Broj LinkedIn pratilaca kompanije |
| time_recorded | Vreme kada je podatak evidentiran |

Tabela omogucava analizu velicine kompanija i njihove prisutnosti na LinkedIn mrezi. Podaci iz ove tabele mogu se povezati sa tabelom `companies` preko atributa `company_id`.

In [59]:
industries = pd.read_csv('../data/industries.csv')

## 2.7 Tabela industries

Tabela `industries` predstavlja sifarnik industrija prisutnih u skupu podataka. Svakoj industriji dodeljen je jedinstveni identifikator koji se koristi za povezivanje sa ostalim tabelama.
### Opis kolona

| Kolona | Opis |
|----------|----------|
| industry_id | Jedinstveni identifikator industrije |
| industry_name | Naziv industrije |

U tabeli je identifikovano 388 razlicitih industrija.

Tabela ne sadrzi direktne informacije o poslovima ili kompanijama, vec sluzi kao pomocna tabela za povezivanje preko identifikatora industrije i dobijanje tekstualnog naziva industrije.

In [60]:
job_industries = pd.read_csv('../data/job_industries.csv')

## 2.8 Tabela job_industries

Tabela `job_industries` povezuje oglase za posao sa industrijama kojima pripadaju. Jedan oglas moze biti povezan sa vise industrija, zbog cega se isti `job_id` moze pojaviti u vise redova.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| job_id | Identifikator oglasa za posao |
| industry_id | Identifikator industrije |

Tabela predstavlja vezu izmedju oglasa za posao i industrija definisanih u tabeli `industries`. Tekstualni naziv industrije moze se dobiti povezivanjem sa tabelom `industries` preko kolone `industry_id`.

Jedan oglas moze pripadati jednoj ili vise industrija, sto omogucava detaljniji opis oblasti u kojoj se pozicija nalazi.

In [61]:
job_skills = pd.read_csv('../data/job_skills.csv')

## 2.9 Tabela job_skills

Tabela `job_skills` povezuje oglase za posao sa kategorijama vestina koje su relevantne za konkretnu poziciju. Jedan posao moze biti povezan sa vise kategorija vestina.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| job_id | Identifikator oglasa za posao |
| skill_abr | Skracenica kategorije vestine |


Puni nazivi kategorija mogu se dobiti povezivanjem sa tabelom `skills` preko kolone `skill_abr`.

Tabela predstavlja vezu izmedju poslova i kategorija vestina i omogucava identifikaciju oblasti znanja i kompetencija koje su povezane sa pojedinim pozicijama.

In [62]:
salaries = pd.read_csv('../data/salaries.csv')

## 2.10 Tabela salaries

Tabela `salaries` sadrzi informacije o platama koje su povezane sa oglasima za posao. Podaci su organizovani tako da se svaki zapis odnosi na jedan oglas za posao.

### Opis kolona

| Kolona | Opis |
|----------|----------|
| salary_id | Jedinstveni identifikator zapisa o plati |
| job_id | Identifikator oglasa za posao |
| max_salary | Maksimalna plata |
| med_salary | Medijalna plata |
| min_salary | Minimalna plata |
| pay_period | Period isplate (HOURLY, MONTHLY, YEARLY) |
| currency | Valuta |
| compensation_type | Tip kompenzacije |

Tabela omogucava analizu finansijskih aspekata poslova. Nisu svi oglasi povezani sa zapisom u ovoj tabeli, sto ukazuje da deo poslodavaca ne objavljuje informacije o plati.

In [63]:
skills = pd.read_csv('../data/skills.csv')

In [64]:
skills["skill_name"].unique()

array(['Art/Creative', 'Design', 'Advertising', 'Product Management',
       'Distribution', 'Education', 'Training', 'Project Management',
       'Consulting', 'Purchasing', 'Supply Chain', 'Analyst',
       'Health Care Provider', 'Research', 'Science', 'General Business',
       'Customer Service', 'Strategy/Planning', 'Finance', 'Other',
       'Legal', 'Engineering', 'Quality Assurance',
       'Business Development', 'Information Technology', 'Administrative',
       'Production', 'Marketing', 'Public Relations', 'Writing/Editing',
       'Accounting/Auditing', 'Human Resources', 'Manufacturing', 'Sales',
       'Management'], dtype=object)

## 2.11 Tabela skills

Tabela `skills` predstavlja sifarnik kategorija vestina koje se koriste u skupu podataka. Svakoj kategoriji dodeljena je skracenica koja se koristi za povezivanje sa tabelom `job_skills`.
### Opis kolona

| Kolona | Opis |
|----------|----------|
| skill_abr | Skracenica kategorije vestine |
| skill_name | Naziv kategorije vestine |

Tabela sadrzi ukupno 35 kategorija vestina.
Tabela ne sadrzi informacije o konkretnim poslovima, vec sluzi kao pomocna tabela za prevodjenje skracenica iz tabele `job_skills` u citljive nazive kategorija vestina.

# 3. Analiza nedostajucih vrednosti

Pre povezivanja tabela i formiranja konacnog skupa podataka izvrsena je analiza nedostajucih vrednosti. Cilj ove analize je identifikacija atributa koji sadrze veliki procenat nedostajucih podataka i donosenje odluke o njihovom ukljucivanju ili uklanjanju iz daljeg procesa obrade.

## 3.1 Tabela postings

Analiza je pokazala da pojedine kolone sadrze veoma visok procenat nedostajucih vrednosti.

| Kolona | Nedostajuce vrednosti (%) |
|----------|----------:|
| closed_time | 99.13 |
| skills_desc | 98.03 |
| med_salary | 94.93 |
| remote_allowed | 87.69 |
| applies | 81.17 |
| min_salary | 75.94 |
| max_salary | 75.94 |
| currency | 70.87 |
| compensation_type | 70.87 |
| pay_period | 70.87 |
| normalized_salary | 70.87 |

Kolone `closed_time` i `skills_desc` sadrze preko 98% nedostajucih vrednosti i ne predstavljaju pouzdan izvor informacija za dalju analizu. Zbog toga ce biti uklonjene iz daljeg procesa obrade.

Kolona `formatted_experience_level` sadrzi oko 24% nedostajucih vrednosti, sto je znacajno manje u odnosu na prethodno navedene atribute. S obzirom da nivo iskustva predstavlja jednu od najvaznijih karakteristika oglasa za posao, ova kolona ce biti zadrzana i naknadno obradjena odgovarajucom metodom imputacije.

Posebnu paznju zahtevaju atributi povezani sa platama. Iako predstavljaju veoma znacajne informacije, vise od 70% oglasa ne sadrzi podatke o plati.

In [65]:
pd.DataFrame({
    "missing_percent": postings.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

,missing_percent
closed_time,99.133622
skills_desc,98.030666
med_salary,94.929309
remote_allowed,87.689848
applies,81.170619
min_salary,75.944093
max_salary,75.944093
currency,70.873402
compensation_type,70.873402
pay_period,70.873402


## 3.2 Tabela companies

Tabela `companies` pokazuje veoma visok kvalitet podataka.

| Kolona | Nedostajuce vrednosti (%) |
|----------|----------:|
| company_size | 11.33 |
| description | 1.21 |
| zip_code | 0.11 |
| state | 0.09 |
| address | 0.09 |

Sve ostale kolone imaju zanemarljiv broj nedostajucih vrednosti.

Jedina kolona koja zahteva dodatnu obradu jeste `company_size`, pri cemu procenat nedostajucih vrednosti ostaje dovoljno nizak da atribut zadrzi svoju analiticku vrednost.


In [66]:
pd.DataFrame({
    "missing_percent": companies.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

,missing_percent
company_size,11.334941
description,1.213582
zip_code,0.114412
state,0.089895
address,0.089895
name,0.004086
city,0.004086
company_id,0.000000
country,0.000000
url,0.000000


## 3.3 Tabela salaries

Analiza tabele `salaries` pokazala je da se najveci problem ne nalazi u samim kolonama, vec u cinjenici da tabela obuhvata samo deo oglasa iz glavne tabele `postings`.

Tabela sadrzi 40785 zapisa, dok glavna tabela sadrzi 123849 oglasa za posao.

Unutar same tabele procenat nedostajucih vrednosti je relativno nizak:

| Kolona | Nedostajuce vrednosti (%) |
|----------|----------:|
| med_salary | 83.23 |
| min_salary | 16.77 |
| max_salary | 16.77 |

Analizom podataka utvrdjeno je da je atribut `normalized_salary` izracunat na osnovu dostupnih podataka o plati i periodu isplate. Za godisnje plate koristi se prosecna vrednost izmedju minimalne i maksimalne plate, dok se satnice i mesecne plate prevode na godisnji nivo.

Medjutim, s obzirom da veliki broj oglasa ne sadrzi nikakve informacije o plati, atributi povezani sa platama verovatno nece biti korisceni kao ulazne karakteristike za klasterovanje. Umesto toga, mogu se koristiti za naknadnu interpretaciju i analizu dobijenih klastera.

In [67]:
pd.DataFrame({
    "missing_percent": salaries.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

,missing_percent
med_salary,83.234032
max_salary,16.765968
min_salary,16.765968
job_id,0.000000
salary_id,0.000000
pay_period,0.000000
currency,0.000000
compensation_type,0.000000


In [68]:
salary_info = postings[
    ["min_salary", "med_salary", "max_salary", "normalized_salary"]
]

salary_info.notna().any(axis=1).mean() * 100

np.float64(29.12659771172961)

In [69]:
pd.DataFrame({
    "missing_percent": employee_counts.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

,missing_percent
company_id,0.0
employee_count,0.0
follower_count,0.0
time_recorded,0.0


# 4. Analiza atributa za klasterovanje

Nakon analize strukture skupa podataka i nedostajucih vrednosti, izvrsena je dodatna analiza kategorickih atributa i povezanih tabela. Cilj ove faze bio je da se identifikuju atributi koji nose korisne informacije za grupisanje poslova, kao i atributi koji ne doprinose procesu klasterovanja.

## 4.1 Analiza atributa iz tabele postings

Analizirani su atributi koji opisuju osnovne karakteristike oglasa za posao.

Atribut `formatted_work_type` sadrzi sedam razlicitih kategorija (Full-time, Contract, Part-time, Internship i druge) i ne sadrzi nedostajuce vrednosti. Ovaj atribut predstavlja vaznu karakteristiku posla i bice ukljucen u dalju analizu.

Atribut `application_type` opisuje nacin prijavljivanja na posao i sadrzi mali broj kategorija. Kako ne postoje nedostajuce vrednosti, atribut ce biti zadrzan.

Atribut `formatted_experience_level` opisuje nivo iskustva kandidata i predstavlja jednu od najvaznijih karakteristika oglasa. Uprkos prisustvu nedostajucih vrednosti, atribut ce biti zadrzan zbog svoje visoke informativnosti.

Analizom atributa `sponsored` utvrdjeno je da svi oglasi imaju istu vrednost. Zbog nepostojanja varijacije ovaj atribut ne moze doprineti razlikovanju poslova i bice uklonjen iz daljeg procesa.

## 4.2 Analiza atributa kompanija

Iz tabele `companies` analizirani su atributi koji opisuju kompanije.

Atribut `company_size` sadrzi ogranicen broj kategorija i predstavlja znacajnu informaciju o velicini kompanije. Zbog toga ce biti ukljucen u konacni skup podataka.

Analizom atributa `country` utvrdjeno je postojanje 81 razlicite drzave. Iako postoji veliki broj kategorija, drzava predstavlja znacajnu geografsku karakteristiku kompanije i zadrzace se u procesu obrade.

Sa druge strane, atribut `state` sadrzi 788 razlicitih vrednosti, sto predstavlja veoma veliku kardinalnost. Zbog velikog broja kategorija ovaj atribut verovatno nece biti ukljucen u finalni skup podataka.

## 4.3 Analiza industrija

Analiza tabele `job_industries` pokazala je da industrijske informacije postoje za 127125 poslova od ukupno 123849 oglasa, pri cemu pojedinacni posao najcesce pripada samo jednoj industriji, a maksimalno trima industrijama.

Najzastupljenije industrije ukljucuju:

- Hospitals and Health Care
- Retail
- IT Services and IT Consulting
- Financial Services
- Software Development
- Manufacturing
- Banking
- Insurance

Rezultati ukazuju da industrija predstavlja znacajnu karakteristiku posla i da moze doprineti kvalitetnijem grupisanju oglasa.

## 4.4 Analiza kategorija vestina

Analiza tabele `job_skills` pokazala je da informacije o vestinama postoje za 126807 poslova.

Svaki posao je povezan sa jednom do tri kategorije vestina, dok tabela `skills` sadrzi ukupno 35 kategorija. Primeri kategorija ukljucuju Engineering, Information Technology, Finance, Sales i Marketing.

S obzirom da kategorije vestina direktno opisuju kompetencije potrebne za posao, predstavljaju jedan od najvrednijih izvora informacija za klasterovanje i bice ukljucene u finalni skup podataka.

## 4.5 Analiza beneficija

Tabela `benefits` sadrzi informacije za 30023 posla i obuhvata ukupno 12 tipova beneficija.

Najzastupljenije beneficije su:

- 401(k)
- Medical Insurance
- Vision Insurance
- Dental Insurance
- Disability Insurance

Iako beneficije nisu dostupne za sve poslove, predstavljaju korisnu informaciju o uslovima zaposlenja i bice ukljucene kao dodatne karakteristike.

## 4.6 Analiza atributa plata

Podaci o platama dostupni su za 40785 poslova, sto predstavlja priblizno trecinu svih oglasa.

Zbog velikog broja nedostajucih vrednosti atributi povezani sa platama nece biti korisceni kao ulazne karakteristike za klasterovanje. Umesto toga, koristice se za naknadnu interpretaciju i analizu dobijenih klastera.

In [70]:
job_industries["industry_id"].nunique()

422

In [71]:
print(postings["formatted_work_type"].value_counts(), "\n")
print(postings["application_type"].value_counts(), "\n")
print(postings["formatted_experience_level"].value_counts(), "\n")
print(postings["sponsored"].value_counts(), "\n")

formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64 

application_type
OffsiteApply          84607
ComplexOnsiteApply    31049
SimpleOnsiteApply      8192
UnknownApply              1
Name: count, dtype: int64 

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64 

sponsored
0    123849
Name: count, dtype: int64 



In [72]:
print(companies["company_size"].value_counts(),"\n")
print(companies["country"].value_counts(),"\n")
print(companies["state"].nunique(),"\n")

company_size
2.0    4956
1.0    4348
5.0    3918
3.0    3108
4.0    2333
7.0    1953
6.0    1083
Name: count, dtype: int64 

country
US    21635
0       727
GB      620
CA      258
IN      157
      ...  
JO        1
KY        1
VE        1
SC        1
TZ        1
Name: count, Length: 81, dtype: int64 

788 



In [73]:
job_industries.groupby("job_id").size().describe()

count    127125.000000
mean          1.296425
std           0.646097
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
dtype: float64

In [74]:
job_skills.groupby("job_id").size().describe()

count    126807.000000
mean          1.685774
std           0.675907
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max           3.000000
dtype: float64

In [75]:
print(job_skills["job_id"].nunique(),"\n")
print(salaries["job_id"].nunique(),"\n")
print(job_industries["job_id"].nunique(),"\n")
print(benefits["job_id"].nunique(),"\n")

126807 

40785 

127125 

30023 



In [76]:
top_industries = (
    job_industries["industry_id"]
    .value_counts()
    .head(30)
    .reset_index()
)

top_industries.columns = ["industry_id", "count"]

top_industries.merge(
    industries,
    on="industry_id",
    how="left"
)

,industry_id,count,industry_name
0,14,18326,Hospitals and Health Care
1,27,11033,Retail
2,96,10396,IT Services and IT Consulting
3,104,9005,Staffing and Recruiting
4,43,8535,Financial Services
5,4,5091,Software Development
6,25,3689,Manufacturing
7,48,3445,Construction
8,41,2923,Banking
9,42,2673,Insurance


# 5. Formiranje master skupa podataka

Nakon analize svih tabela iz skupa podataka, izvrseno je formiranje jedinstvene tabele koja predstavlja osnovu za dalje korake preprocesiranja i klasterovanja.

Kao centralna tabela koriscena je tabela `postings`, pri cemu svaki red predstavlja jedan oglas za posao. Dodatne informacije iz ostalih tabela povezane su koriscenjem odgovarajucih identifikatora (`job_id` i `company_id`).

U master tabelu ukljuceni su podaci o kompaniji, velicini kompanije, broju zaposlenih i pratilaca, vestinama koje se zahtevaju za posao, industrijama kojima posao pripada, kao i pogodnostima koje poslodavac nudi.

Tabele koje sadrze visestruke vrednosti po poslu (`job_skills`, `job_industries` i `benefits`) transformisane su primenom one-hot encoding postupka, cime je svaka kategorija predstavljena posebnom binarnom kolonom.

Prilikom izbora atributa analizirani su procenat nedostajucih vrednosti, zastupljenost pojedinih kategorija i mogucnost njihove primene u procesu klasterovanja. Za industrije su zadrzane 75 najzastupljenijih kategorija, dok su sve kategorije vestina i pogodnosti ukljucene u konacni skup podataka.

Na ovaj nacin formiran je jedinstven skup podataka sa 123849 redova i 157 atributa, koji predstavlja polaznu tacku za dalje ciscenje podataka, obradu nedostajucih vrednosti i primenu algoritama klasterovanja.

In [77]:
master = postings.copy()

In [78]:
companies_subset = companies[["company_id", "company_size", "country"]]

In [79]:
master = master.merge(companies_subset, on="company_id", how="left")

In [80]:
employee_counts_latest = (employee_counts.sort_values("time_recorded").drop_duplicates(
        subset="company_id",
        keep="last"
    )
)

In [81]:
employee_counts_subset = employee_counts_latest[["company_id", "employee_count", "follower_count"]]

In [82]:
master = master.merge(employee_counts_subset, on="company_id", how="left")
master.shape

(123849, 35)

In [83]:
#ONE-HOT ENCODING
job_skills_encoded = pd.crosstab(
    job_skills["job_id"],
    job_skills["skill_abr"]
)
job_skills_encoded.shape

(126807, 35)

In [84]:
skill_cols = job_skills_encoded.columns.tolist()
master = master.merge(job_skills_encoded, on="job_id", how="left")

In [85]:
master[skill_cols] = master[skill_cols].fillna(0)
master.shape

(123849, 70)

In [86]:
benefits_encoded = pd.crosstab(benefits["job_id"], benefits["type"])

benefit_cols = benefits_encoded.columns.tolist()

master = master.merge(benefits_encoded, on="job_id", how="left")

master[benefit_cols] = master[benefit_cols].fillna(0)

In [88]:
job_industries_named = job_industries.merge(industries, on="industry_id", how="left")

In [89]:
for n in [30, 50, 75, 100]:
    top_industries = (
        job_industries_named["industry_name"].value_counts().head(n).index)

    covered_jobs = (
        job_industries_named[
            job_industries_named["industry_name"].isin(top_industries)
        ]["job_id"]
        .nunique()
    )

    print(
        f"Top {n}: "
        f"{covered_jobs} poslova "
        f"({covered_jobs / postings['job_id'].nunique() * 100:.2f}%)"
    )

Top 30: 97344 poslova (78.60%)
Top 50: 110659 poslova (89.35%)
Top 75: 119143 poslova (96.20%)
Top 100: 122077 poslova (98.57%)


In [90]:
top_75_industries = (job_industries_named["industry_name"].value_counts().head(75).index)

job_industries_top = job_industries_named[job_industries_named["industry_name"].isin(top_75_industries)]

industries_encoded = pd.crosstab(job_industries_top["job_id"], job_industries_top["industry_name"])

industry_cols = industries_encoded.columns.tolist()

master = master.merge(industries_encoded, on="job_id", how="left")

master[industry_cols] = master[industry_cols].fillna(0)

In [91]:
master.shape

(123849, 157)

In [92]:
master.to_csv("../data/master_raw.csv", index=False)